# GSEA and TGF-beta/LOX/EMT Axis Analysis

This notebook tests whether thymic aging is associated with TGF-beta signaling, EMT-like programs, inflammatory signaling, and LOX-family activity in annotated thymus scRNA-seq data.

The main analyses are cell-type-specific GSEA, fibroblast-level signature scoring, age-stratified correlation testing, and an optional TEC trajectory analysis using PAGA.

## 1. Setup

We import analysis libraries, install `gseapy` if needed, set random seeds for reproducibility, and define project paths. Figures are saved into dedicated folders so GSEA, signature, and pseudotime outputs are easy to inspect.

In [ ]:
from pathlib import Path
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy import sparse
from scipy.stats import norm, spearmanr

try:
    import gseapy as gp
except ImportError:
    print("gseapy is not installed. Installing with pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gseapy"])
    import gseapy as gp

np.random.seed(0)
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white")
sns.set_theme(style="whitegrid", context="paper")

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
GSEA_FIG_DIR = FIGURES_DIR / "GSEA_TGFb"
SIGNATURE_FIG_DIR = FIGURES_DIR / "TGFb_LOX_EMT_signatures"
PSEUDOTIME_FIG_DIR = FIGURES_DIR / "pseudotime"

for path in [PROCESSED_DIR, GSEA_FIG_DIR, SIGNATURE_FIG_DIR, PSEUDOTIME_FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_GENE_SETS = [
    "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
]
CELL_TYPES_FOR_GSEA = ["cTEC", "mTEC", "Fibroblasts"]
AGE_PALETTE = ["#0072B2", "#D55E00"]

## 2. Load Annotated Thymus Data

We load the annotated AnnData object from notebook 04. When `adata.raw` exists, it is used as the expression source because it should contain normalized log expression before scaling and highly variable gene subsetting.

In [ ]:
input_path = PROCESSED_DIR / "thymus_annotated.h5ad"
if not input_path.exists():
    raise FileNotFoundError(f"Missing {input_path}. Run notebook 04 first.")

adata = sc.read_h5ad(input_path)
expr_adata = adata.raw.to_adata() if adata.raw is not None else adata

if "cell_type" not in adata.obs.columns:
    raise KeyError("Expected adata.obs['cell_type']. Run notebook 04 first.")

age_candidates = ["age_group", "Age", "age", "age_group_label", "donor_age", "development_stage"]
AGE_KEY = next((column for column in age_candidates if column in adata.obs.columns), None)
if AGE_KEY is None:
    raise KeyError("No age-group metadata column found. Create adata.obs['age_group'] before running this notebook.")

print(f"Using age metadata column: {AGE_KEY}")
print(adata.obs[AGE_KEY].value_counts(dropna=False))
adata

## 3. Helper Functions

These helper functions handle common practical issues: detecting young/old labels, extracting expression from sparse or dense matrices, mapping requested genes to the dataset's capitalization, and comparing correlations between age groups using Fisher's z-transformation.

In [ ]:
def classify_age_labels(values):
    labels = [str(value) for value in pd.Series(values).dropna().unique()]
    young_terms = ["young", "juvenile", "neonate", "newborn", "fetal", "foetal", "pcw", "4w", "1m"]
    old_terms = ["old", "aged", "adult", "elder", "elderly", "18m", "24m"]
    young = [label for label in labels if any(term in label.lower() for term in young_terms)]
    old = [label for label in labels if any(term in label.lower() for term in old_terms)]
    if not young or not old:
        sorted_labels = sorted(labels)
        if len(sorted_labels) >= 2:
            young = [sorted_labels[0]]
            old = [sorted_labels[-1]]
    return young, old


def expression_vector(adata_like, gene):
    values = adata_like[:, gene].X
    if sparse.issparse(values):
        values = values.toarray()
    return np.asarray(values).ravel()


def map_genes(requested_genes, var_names):
    lookup = {gene.upper(): gene for gene in pd.Index(var_names).astype(str)}
    return {gene: lookup.get(gene.upper()) for gene in requested_genes}


def fisher_z_compare(r_young, n_young, r_old, n_old):
    if n_young <= 3 or n_old <= 3 or np.isnan(r_young) or np.isnan(r_old):
        return np.nan, np.nan
    r_young = np.clip(r_young, -0.999999, 0.999999)
    r_old = np.clip(r_old, -0.999999, 0.999999)
    z = (np.arctanh(r_old) - np.arctanh(r_young)) / np.sqrt(1 / (n_old - 3) + 1 / (n_young - 3))
    p = 2 * (1 - norm.cdf(abs(z)))
    return z, p


young_labels, old_labels = classify_age_labels(adata.obs[AGE_KEY])
print("Young labels:", young_labels)
print("Old labels:", old_labels)

## Analysis 1 - Gene Set Enrichment Analysis

For cTECs, mTECs, and fibroblasts, we rank genes by old-versus-young differential expression and run preranked GSEA against selected MSigDB Hallmark pathways. The selected sets focus on TGF-beta signaling, EMT, inflammatory response, and TNF-alpha/NF-kB signaling.

### 4. Retrieve MSigDB Hallmark Gene Sets

`gseapy.msigdb_gsets()` is used to query available MSigDB collections. We then load the Hallmark collection and keep only the four target pathways. If the exact API differs by `gseapy` version, the fallback uses Enrichr's Hallmark 2020 library names.

In [ ]:
def load_target_gene_sets():
    available = gp.msigdb_gsets()
    print("Retrieved MSigDB collection list with gseapy.msigdb_gsets().")

    hallmark_candidates = [
        "HALLMARK",
        "h.all.v2023.1.Hs.symbols.gmt",
        "h.all.v2023.2.Hs.symbols.gmt",
        "MSigDB_Hallmark_2020",
    ]

    gene_sets = None
    errors = []
    for candidate in hallmark_candidates:
        try:
            gene_sets = gp.get_library(name=candidate)
            print(f"Loaded gene sets with gp.get_library(name={candidate!r}).")
            break
        except Exception as exc:
            errors.append(f"{candidate}: {exc}")

    if gene_sets is None:
        raise RuntimeError("Could not load Hallmark gene sets. Attempts: " + " | ".join(errors))

    normalized = {name.upper().replace(" ", "_"): genes for name, genes in gene_sets.items()}
    selected = {}
    for target in TARGET_GENE_SETS:
        matches = [name for name in normalized if name.endswith(target) or name == target]
        if matches:
            selected[target] = normalized[matches[0]]

    missing = [target for target in TARGET_GENE_SETS if target not in selected]
    if missing:
        raise KeyError(f"Could not find target gene sets: {missing}")

    return selected


target_gene_sets = load_target_gene_sets()
{name: len(genes) for name, genes in target_gene_sets.items()}

### 5. Run Cell-Type-Specific GSEA

Each cell type is analyzed separately so pathway changes are not diluted by unrelated populations. Gene ranking uses Scanpy's Wilcoxon test comparing old labels against young labels.

In [ ]:
def make_gsea_bubble(results_df, cell_type, output_path):
    if results_df.empty:
        print(f"No GSEA results to plot for {cell_type}.")
        return
    plot_df = results_df.copy()
    padj_col = "FDR q-val" if "FDR q-val" in plot_df.columns else "padj"
    nes_col = "NES"
    term_col = "Term" if "Term" in plot_df.columns else "gene_set"
    plot_df["minus_log10_padj"] = -np.log10(plot_df[padj_col].astype(float).clip(lower=1e-300))
    if "Tag %" in plot_df.columns:
        plot_df["gene_set_size"] = plot_df["Tag %"].astype(str).str.extract(r"(\d+)").astype(float).fillna(25)
    else:
        plot_df["gene_set_size"] = 25

    plt.figure(figsize=(7, 4))
    sns.scatterplot(
        data=plot_df,
        x=nes_col,
        y="minus_log10_padj",
        size="gene_set_size",
        hue=nes_col,
        palette="vlag",
        sizes=(80, 500),
        edgecolor="black",
        linewidth=0.4,
    )
    plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
    for _, row in plot_df.iterrows():
        plt.text(row[nes_col], row["minus_log10_padj"] + 0.05, str(row[term_col]).replace("HALLMARK_", ""), fontsize=8, ha="center")
    plt.title(f"GSEA old vs young: {cell_type}")
    plt.xlabel("Normalized enrichment score (NES)")
    plt.ylabel("-log10 adjusted p-value")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


gsea_results = {}

for cell_type in CELL_TYPES_FOR_GSEA:
    cell_mask = adata.obs["cell_type"].astype(str).str.lower() == cell_type.lower()
    if cell_mask.sum() < 20:
        print(f"Skipping {cell_type}: fewer than 20 cells.")
        continue

    subset_expr = expr_adata[cell_mask].copy()
    subset_obs = adata.obs.loc[cell_mask].copy()
    subset_expr.obs[AGE_KEY] = subset_obs[AGE_KEY].astype(str).values

    selected_mask = subset_expr.obs[AGE_KEY].isin(young_labels + old_labels)
    subset_expr = subset_expr[selected_mask].copy()
    subset_expr.obs["young_old"] = np.where(subset_expr.obs[AGE_KEY].isin(old_labels), "old", "young")

    if subset_expr.obs["young_old"].nunique() < 2:
        print(f"Skipping {cell_type}: young/old groups are not both present.")
        continue

    sc.tl.rank_genes_groups(subset_expr, groupby="young_old", groups=["old"], reference="young", method="wilcoxon")
    rank_df = sc.get.rank_genes_groups_df(subset_expr, group="old")
    rank_df = rank_df.dropna(subset=["names", "scores"]).drop_duplicates("names")
    ranking = rank_df[["names", "scores"]].sort_values("scores", ascending=False)

    pre_res = gp.prerank(
        rnk=ranking,
        gene_sets=target_gene_sets,
        min_size=5,
        max_size=500,
        permutation_num=1000,
        seed=0,
        outdir=None,
        verbose=False,
    )

    result_df = pre_res.res2d.copy()
    result_df.to_csv(PROCESSED_DIR / f"GSEA_results_{cell_type}.csv", index=False)
    gsea_results[cell_type] = result_df
    make_gsea_bubble(result_df, cell_type, GSEA_FIG_DIR / f"GSEA_bubble_{cell_type}.png")

gsea_results.keys()

## Analysis 2 - TGF-beta Pathway in Context of LOX

In fibroblasts, we score three per-cell signatures: TGF-beta signaling, EMT, and LOX-family activity. We then visualize score relationships, compute Spearman correlations, and test whether correlations differ between young and old cells using Fisher's z-transformation.

In [ ]:
signature_requests = {
    "TGFb_score": ["Tgfb1", "Tgfb2", "Smad2", "Smad3", "Smad4", "Tgfbr1", "Tgfbr2"],
    "EMT_score": ["Vim", "Snai1", "Snai2", "Twist1", "Cdh2", "Fn1"],
    "LOX_activity_score": ["Lox", "Loxl1", "Loxl2", "Loxl3", "Loxl4"],
}

signature_genes = {}
for score_name, genes in signature_requests.items():
    mapped = map_genes(genes, expr_adata.var_names)
    present = [gene for gene in mapped.values() if gene is not None]
    signature_genes[score_name] = present
    print(f"{score_name}: {present}")

fibro_mask = adata.obs["cell_type"].astype(str).str.lower() == "fibroblasts"
if fibro_mask.sum() == 0:
    raise ValueError("No fibroblasts found in adata.obs['cell_type'].")

fibro = expr_adata[fibro_mask].copy()
fibro.obs[AGE_KEY] = adata.obs.loc[fibro.obs_names, AGE_KEY].astype(str)

for score_name, genes in signature_genes.items():
    if len(genes) < 2:
        raise ValueError(f"Too few genes found for {score_name}: {genes}")
    sc.tl.score_genes(fibro, gene_list=genes, score_name=score_name, random_state=0)

score_columns = ["TGFb_score", "EMT_score", "LOX_activity_score"]
fibro.obs[score_columns + [AGE_KEY]].head()

In [ ]:
score_df = fibro.obs[score_columns + [AGE_KEY]].copy()

pair_grid = sns.pairplot(
    score_df,
    vars=score_columns,
    hue=AGE_KEY,
    palette=AGE_PALETTE,
    corner=False,
    plot_kws={"s": 10, "alpha": 0.45, "edgecolor": "none"},
    diag_kind="kde",
)
pair_grid.fig.suptitle("Fibroblast TGF-beta, EMT, and LOX signature scores", y=1.02)
pair_grid.fig.savefig(SIGNATURE_FIG_DIR / "fibroblast_signature_scatter_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
corr_matrix = score_df[score_columns].corr(method="spearman")
corr_matrix.to_csv(PROCESSED_DIR / "fibroblast_TGFb_LOX_EMT_spearman_matrix.csv")

plt.figure(figsize=(4.5, 4))
sns.heatmap(corr_matrix, vmin=-1, vmax=1, cmap="vlag", annot=True, fmt=".2f", square=True)
plt.title("Fibroblast Spearman correlations")
plt.tight_layout()
plt.savefig(SIGNATURE_FIG_DIR / "fibroblast_signature_spearman_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

corr_matrix

In [ ]:
comparison_rows = []
for score_a, score_b in [("TGFb_score", "LOX_activity_score"), ("LOX_activity_score", "EMT_score"), ("TGFb_score", "EMT_score")]:
    young_df = score_df[score_df[AGE_KEY].isin(young_labels)]
    old_df = score_df[score_df[AGE_KEY].isin(old_labels)]
    r_young, p_young = spearmanr(young_df[score_a], young_df[score_b]) if len(young_df) > 2 else (np.nan, np.nan)
    r_old, p_old = spearmanr(old_df[score_a], old_df[score_b]) if len(old_df) > 2 else (np.nan, np.nan)
    z_stat, p_diff = fisher_z_compare(r_young, len(young_df), r_old, len(old_df))
    comparison_rows.append({
        "score_a": score_a,
        "score_b": score_b,
        "n_young": len(young_df),
        "n_old": len(old_df),
        "rho_young": r_young,
        "p_young": p_young,
        "rho_old": r_old,
        "p_old": p_old,
        "fisher_z_old_minus_young": z_stat,
        "p_correlation_difference": p_diff,
    })

correlation_comparison = pd.DataFrame(comparison_rows)
correlation_comparison.to_csv(PROCESSED_DIR / "fibroblast_signature_correlation_age_comparison.csv", index=False)
correlation_comparison

## Analysis 3 - Optional TEC PAGA Trajectory

PAGA provides a graph abstraction of cluster relationships. Here we run it on TEC populations if enough cTEC/mTEC cells are available. The biological question is whether TECs appear to move toward EMT-like or fibrotic states with age, and whether that trajectory is associated with higher LOX activity.

In [ ]:
tec_mask = adata.obs["cell_type"].astype(str).isin(["cTEC", "mTEC"])

if tec_mask.sum() < 50:
    print(f"Skipping PAGA: only {tec_mask.sum()} cTEC/mTEC cells found.")
else:
    tec = expr_adata[tec_mask].copy()
    tec.obs["cell_type"] = adata.obs.loc[tec.obs_names, "cell_type"].astype(str)
    tec.obs[AGE_KEY] = adata.obs.loc[tec.obs_names, AGE_KEY].astype(str)

    lox_genes = signature_genes["LOX_activity_score"]
    sc.tl.score_genes(tec, gene_list=lox_genes, score_name="LOX_activity_score", random_state=0)

    sc.pp.highly_variable_genes(tec, n_top_genes=min(3000, tec.n_vars))
    tec = tec[:, tec.var["highly_variable"]].copy()
    sc.pp.scale(tec, max_value=10)
    sc.tl.pca(tec, n_comps=min(30, tec.n_obs - 1), svd_solver="arpack", random_state=0)
    sc.pp.neighbors(tec, n_neighbors=15, n_pcs=min(20, tec.obsm["X_pca"].shape[1]))
    sc.tl.leiden(tec, resolution=0.5, key_added="paga_cluster", random_state=0)
    sc.tl.paga(tec, groups="paga_cluster")
    sc.pl.paga(tec, color=["cell_type", AGE_KEY], show=False)
    plt.savefig(PSEUDOTIME_FIG_DIR / "TEC_PAGA_graph.png", dpi=300, bbox_inches="tight")
    plt.show()

    sc.tl.umap(tec, init_pos="paga", random_state=0)
    sc.pl.umap(tec, color=["cell_type", AGE_KEY, "LOX_activity_score"], ncols=3, frameon=False, show=False)
    plt.savefig(PSEUDOTIME_FIG_DIR / "TEC_PAGA_UMAP_LOX_score.png", dpi=300, bbox_inches="tight")
    plt.show()

    tec.write(PROCESSED_DIR / "TEC_PAGA_LOX_trajectory.h5ad")

## Interpretation: TGF-beta -> LOX -> EMT Axis

After running the notebook, interpret support for the TGF-beta -> LOX -> EMT axis using three layers of evidence. First, check whether GSEA shows positive NES and significant adjusted p-values for `HALLMARK_TGF_BETA_SIGNALING` and `HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION` in the same cell type, especially fibroblasts or TEC subsets. Second, check whether fibroblast TGF-beta, LOX activity, and EMT scores are positively correlated, and whether those correlations strengthen in old cells by Fisher's z-test. Third, inspect whether the TEC PAGA/UMAP trajectory shows increasing LOX activity along a path toward aged or EMT-like states.

The axis is supported if old fibroblasts or TECs show enriched TGF-beta and EMT pathways, higher LOX activity, and positive TGF-beta/LOX/EMT score correlations. It is contradicted if TGF-beta or EMT enrichment is absent, LOX activity is uncoupled from EMT scores, or the strongest signal occurs in unrelated immune populations rather than stromal compartments. The strongest cell type should be named from the GSEA table with the largest positive NES and lowest adjusted p-value, then cross-checked against the fibroblast signature correlations and TEC trajectory plots.